# EDA Olist: vì sao review xấu tăng vọt đầu năm 2018?

Giám đốc vận hành thấy tỉ lệ review 1–2 sao tăng mạnh trong quý 1/2018 và hỏi: chuyện gì xảy ra, và
nên làm gì? Bạn có hai ngày. Kết quả nộp là notebook này (EDA có cấu trúc) và `MEMO.md` một trang.

Ba hàm được chấm tự động; phần còn lại của notebook là của bạn: thêm cell `explore` tuỳ ý.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) là chỗ phân tích. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/09-eda-storytelling/exercises/advanced/01-olist-eda-memo -q`

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

## 1. Bảng phân tích

Một dòng mỗi đơn. Ghi lại số dòng mất ở mỗi bước lọc.

In [ ]:
def order_table(orders: pd.DataFrame, items: pd.DataFrame, reviews: pd.DataFrame,
                customers: pd.DataFrame, sellers: pd.DataFrame) -> pd.DataFrame:
    """Bảng phân tích: một dòng mỗi đơn giao xong có review (định nghĩa dùng chung của track).

    Cột: order_id, purchase_month (Period tháng của order_purchase_timestamp), customer_state,
    seller_state (seller của item có order_item_id == 1), n_items (số dòng item), n_sellers (số
    seller khác nhau), price (tổng price), freight (tổng freight_value), delivery_days (giao − đặt,
    ngày, float), promised_days (ngày hẹn − đặt, ngày, float), late, bad.
    """
    # TODO
    raise NotImplementedError


def rate_by(table: pd.DataFrame, factor: str, target: str = "bad", min_n: int = 30,
            z: float = 1.96) -> pd.DataFrame:
    """Tỉ lệ target theo từng giá trị của factor, kèm khoảng Wilson.

    Index: giá trị của factor (tên index = factor). Cột: n, rate, ci_low, ci_high, lift.
    lift = rate / tỉ lệ target trên TOÀN BỘ table. Bỏ nhóm có n < min_n. Sắp rate giảm dần.
    """
    # TODO
    raise NotImplementedError


def decompose_change(table: pd.DataFrame, period_col: str, a, b, segment: str = "late",
                     target: str = "bad") -> dict:
    """Tách thay đổi tỉ lệ target từ kỳ a sang kỳ b thành phần mix và phần rate.

    Với w_{p,s} là tỉ trọng segment s trong kỳ p và r_{p,s} là tỉ lệ target của segment s trong kỳ p:
      total = rate(b) − rate(a)
      mix   = sum_s (w_{b,s} − w_{a,s}) · r_{a,s}
      rate  = sum_s w_{b,s} · (r_{b,s} − r_{a,s})
    (mix + rate = total). Segment phải có mặt ở cả hai kỳ. Trả {"total", "mix", "rate"}.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# explore
table = order_table(orders, items, reviews, customers, sellers)
print(len(table))
table.head()

## 2. Hiện tượng

Review xấu và giao trễ theo tháng. Bỏ tháng chưa chín (đơn mua sau 2018-08).

In [ ]:
# explore
monthly = table.groupby("purchase_month")[["bad", "late"]].mean()
monthly.loc["2017-01":"2018-08"].round(3)

## 3. Yếu tố đi cùng review xấu

Dùng `rate_by` với nhiều factor (trễ, bang, số item, số seller...). Đọc n và khoảng tin cậy trước khi đọc thứ hạng.

In [ ]:
# explore
rate_by(table, "customer_state").head(10)

## 4. Tách mix và rate

Phần tăng review xấu từ tháng 1 sang tháng 3/2018 đến từ việc có nhiều đơn trễ hơn (mix), hay đơn trễ/đúng hẹn đều bị chấm tệ hơn (rate)?

In [ ]:
# explore
decompose_change(table, "purchase_month", pd.Period("2018-01", "M"), pd.Period("2018-03", "M"))

## 5. Phân tích của bạn

Thêm cell `explore`: đi tiếp vào giao trễ (tuyến bang seller → bang khách, số ngày hẹn, category...), kiểm Simpson, vẽ 2–3 biểu đồ sẽ đưa vào memo (tiêu đề là kết luận). Rồi viết `MEMO.md`.

In [ ]:
# explore: phân tích của bạn